# Planowanie linii metra we Wrocławiu

Notebook jest szkicem projektu badawczo-inżynierskiego: chcemy znaleźć wariant przebiegu metra, który przy długości i liczbie stacji jak warszawska M1 przechodzi przez centrum Wrocławia, unika obszarów zalewowych i obsługuje jak największy popyt.

Założenie bazowe: jedna linia ma 23,1 km i 21 stacji. Scenariusze `1`, `2`, `3` oznaczają odpowiednio sieć z jedną, dwiema i trzema liniami. Koszt kilometra, promień dojścia do stacji, kara za obszary zalewowe i mnożnik geologiczny są parametrami, które można zmieniać.

## Logika projektu

1. Zbieramy popyt: najlepiej z demografii SIP Wrocławia, a alternatywnie z lokali wyborczych i liczby oddanych głosów.
2. Zamieniamy popyt na ważone punkty: centroid/reprezentatywny punkt regionu, adresu albo obwodu.
3. Wczytujemy obszary zakazane: przede wszystkim realne obszary zagrożenia powodziowego MZP/ISOK, jeśli są dostępne lokalnie. Rzeki i wody powierzchniowe są osobną warstwą bariery komunikacyjnej, nie zakazem dla metra.
4. Popyt leżący w obszarze zakazanym przesuwamy do najbliższego wolnego miejsca, bez zmiany wagi.
5. Tworzymy centra regionalne i kandydatów na stacje.
6. Formułujemy problem jako orienteering / prize-collecting TSP: wybierz i uporządkuj kandydatów tak, żeby zmieścić się w 23,1 km i zebrać jak najwięcej popytu.
7. Rozwiązujemy problem heurystyką greedy insertion + 2-opt, a warianty oceniamy po obsłużonym popycie, przejściu przez strefy zakazane, geologii i koszcie.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

import wro_metro as wm

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT

## Źródła danych

Najbardziej wiarygodny wariant popytu to demografia SIP Wrocławia, bo jest przestrzenna i aktualna. Lokale wyborcze są dobrym proxy aktywności, ale nie powinny być jedyną miarą populacji: głosy zależą od frekwencji, wieku, granic obwodów i typu wyborów.


In [ ]:
sources = pd.DataFrame([
    {
        "warstwa": "Benchmark metra",
        "zastosowanie": "23,1 km i 21 stacji jako ograniczenia bazowe",
        "url": "https://metro.waw.pl/metro-warszawskie/linia-m1/przebieg/",
    },
    {
        "warstwa": "Demografia SIP Wrocławia 1998-2025",
        "zastosowanie": "najlepszy bazowy popyt: ludność według rejonów statystycznych i urbanistycznych",
        "url": "https://geoportal.wroclaw.pl/zasoby/",
    },
    {
        "warstwa": "Granice osiedli i adresy SIP",
        "zastosowanie": "centra osiedli, agregacje, walidacja adresów",
        "url": "https://geoportal.wroclaw.pl/zasoby/",
    },
    {
        "warstwa": "Wody powierzchniowe SIP",
        "zastosowanie": "tymczasowe proxy przeszkód wodnych, nie zamiennik map zalewowych",
        "url": "https://geoportal.wroclaw.pl/zasoby/",
    },
    {
        "warstwa": "Mapy zagrożenia powodziowego ISOK/Wody Polskie",
        "zastosowanie": "obszary zakazane albo silnie karane w przebiegu trasy",
        "url": "https://wody.isok.gov.pl/imap_kzgw/?gpmap=gpMZP",
    },
    {
        "warstwa": "Udokumentowane odwierty geologiczne Wrocławia",
        "zastosowanie": "opcjonalne dane do zbudowania realnego mnożnika kosztu; bez nich model używa neutralnego 1.0",
        "url": "https://www.geoportal.wroclaw.pl/mapy/odwierty/",
    },
    {
        "warstwa": "GUS BDL API",
        "zastosowanie": "dane kontrolne i agregaty demograficzne, mniej szczegółowe przestrzennie",
        "url": "https://api.stat.gov.pl/Home/BdlApi",
    },
])
sources

## Wczytanie danych

Notebook najpierw próbuje użyć oficjalnych paczek SIP Wrocławia z `data/raw`. Jeżeli ich nie ma, przełącza się na mały zestaw demonstracyjny. Popyt rysujemy jako obszary demograficzne, a algorytm używa ich punktów reprezentatywnych tylko jako technicznej reprezentacji do liczenia odległości. Dane zalewowe z ISOK najlepiej dodać jako `data/raw/flood_zones.geojson`. Wody powierzchniowe nie są traktowane jako zakaz dla metra: są osobną warstwą bariery komunikacyjnej, której przecięcie może poprawić połączenia między brzegami.


In [ ]:
config = wm.MetroConfig(
    cost_per_km_mln=650.0,
    walk_radius_m=850.0,
    angle_step_deg=2.0,
    forbidden_penalty_per_km=2_500.0,
    river_crossing_bonus_per_km=600.0,
    transfer_bonus_per_interchange=35_000.0,
    interchange_radius_m=450.0,
    line_overlap_penalty_per_km=75_000.0,
    parallel_line_buffer_m=450.0,
)

raw_dir = PROJECT_ROOT / "data" / "raw"
demography_zip = raw_dir / "dem-rejurb-rejstat-shp.zip"

if demography_zip.exists():
    population_layer = wm.load_wroclaw_demography(raw_dir, year=2025, unit="REJSTAT")
    demand_areas = wm.demand_areas_from_polygons(population_layer, weight_col="SUMA")
    demand_raw = wm.to_demand_points(population_layer, weight_col="SUMA")
    demand_raw["name"] = "rejon_" + demand_raw["REJON"].astype(str)
    osiedla = wm.load_wroclaw_osiedla(raw_dir)
    centre_area = osiedla[osiedla["NAZWAOSIED"].eq("Stare Miasto")].copy()
    data_mode = "oficjalna demografia SIP Wrocławia 2025"
else:
    demand_raw = wm.demo_demand()
    demand_areas = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    centre_area = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    data_mode = "dane demonstracyjne"

centres = wm.demo_centres()
if not centre_area.empty:
    stare_miasto_point = centre_area.geometry.iloc[0].representative_point()
    mask = centres["role"].eq("required_city_centre")
    centres.loc[mask, "name"] = "Stare Miasto"
    centres.loc[mask, "geometry"] = [stare_miasto_point]

forbidden_from_raw = wm.load_forbidden_zones_from_raw(raw_dir)
if forbidden_from_raw is not None:
    flood_zones = forbidden_from_raw
    forbidden_mode = sorted(flood_zones["risk"].dropna().unique().tolist())
else:
    flood_zones = gpd.GeoDataFrame({"risk": []}, geometry=[], crs=demand_raw.crs)
    forbidden_mode = ["brak mapy zalewowej ISOK w data/raw"]

water_crossings = wm.load_water_crossing_layer(raw_dir)
if water_crossings is None:
    water_crossings = wm.demo_flood_zones()
    water_mode = "demo river proxy"
else:
    water_mode = "wody powierzchniowe SIP jako bariera/crossing layer"

geology_from_raw = wm.load_geology_cost_layer_from_raw(raw_dir)
if geology_from_raw is not None:
    geology = geology_from_raw
    geology_mode = "realna warstwa kosztu/geologii z data/raw"
else:
    geology = gpd.GeoDataFrame({"cost_factor": []}, geometry=[], crs=demand_raw.crs)
    geology_mode = "brak realnej warstwy geologicznej; mnożnik kosztu = 1.0"

if not demand_areas.empty:
    regional_clusters = wm.regional_clusters_from_demand_areas(demand_areas, k=8)
    regional_centres = wm.regional_centres_from_clusters(regional_clusters)
else:
    regional_clusters = gpd.GeoDataFrame(geometry=[], crs=demand_raw.crs)
    regional_centres = wm.regional_centres_from_demand(demand_raw, k=8)

print(f"Tryb danych: {data_mode}")
print(f"Warstwy zakazane/proxy: {forbidden_mode}")
print(f"Warstwa rzek/wody: {water_mode}")
print(f"Geologia/koszt: {geology_mode}")
print(f"Centrum wymuszone: {centres.loc[centres['role'].eq('required_city_centre'), 'name'].iloc[0]}")
print(f"Długość linii: {config.length_m/1000:.1f} km")
print(f"Liczba stacji: {config.station_count}")
print(f"Średni rozstaw stacji: {config.station_spacing_m:.0f} m")

display_cols = [col for col in ["name", "REJON", "population"] if col in demand_raw.columns]
demand_raw[display_cols].sort_values("population", ascending=False).head(10)

In [ ]:
# Przykład podmiany na dane realne po pobraniu plików do data/raw:
# py -3.11 scripts\download_wroclaw_data.py
# population_layer = wm.load_wroclaw_demography(PROJECT_ROOT / "data/raw", year=2025, unit="REJSTAT")
# demand_areas = wm.demand_areas_from_polygons(population_layer, weight_col="SUMA")
# demand_raw = wm.to_demand_points(population_layer, weight_col="SUMA")
# flood_zones = wm.load_forbidden_zones_from_raw(PROJECT_ROOT / "data/raw")
# water_crossings = wm.load_water_crossing_layer(PROJECT_ROOT / "data/raw")
#
# polling = wm.load_polling_places_csv(PROJECT_ROOT / "data/raw/polling_places.csv", votes_col="votes")
# demand_raw = polling
#
# Jeżeli CSV z lokalami wyborczymi nie ma współrzędnych:
# api_key = os.environ["GOOGLE_MAPS_API_KEY"]
# polling_table = pd.read_csv(PROJECT_ROOT / "data/raw/polling_places.csv")
# polling_geocoded = wm.geocode_google_addresses(polling_table, address_col="address", api_key=api_key)
# polling_geocoded.to_csv(PROJECT_ROOT / "data/processed/polling_places_geocoded.csv", index=False)

None

### Ważne rozróżnienia danych

- Obszary zalewowe: w notebooku pojawią się dopiero wtedy, gdy w `data/raw` znajdzie się realna warstwa MZP/ISOK, np. `flood_zones.geojson`. Obecnie jej nie ma, więc nie rysujemy fałszywej warstwy zalewowej.
- Rzeka i wody powierzchniowe: pochodzą z SIP Wrocławia i są używane jako warstwa bariery komunikacyjnej oraz potencjalnej premii za przeprawę metrem, nie jako teren zakazany.
- Mnożnik kosztu geologicznego: poprzednio był syntetycznym placeholderem. Teraz w scenariuszu bazowym jest neutralny (`1.0`), dopóki nie dodamy realnej warstwy `geology.geojson` albo `cost_zones.geojson` z kolumną `cost_factor`.
- Centra regionalne: nie są ręcznie wpisanymi punktami. Powstają z klastrów ważonych populacją na obszarach demograficznych SIP.


## Obszary popytu, zalewowość i rzeka

Założenie: popyt mieszkaniowy analizujemy jako obszary demograficzne, a do optymalizacji przekazujemy ich punkty reprezentatywne. Jeżeli punkt popytu trafi w obszar realnego ryzyka zalewowego, nie usuwamy go z modelu, tylko przesuwamy jego wagę do najbliższego wolnego miejsca. Rzeka nie jest obszarem zakazanym dla metra: potraktowana jest jako bariera komunikacyjna, której przecięcie może dać premię, bo poprawia relacje między brzegami.


In [ ]:
demand = wm.relocate_demand_from_forbidden(demand_raw, flood_zones, config=config)
if regional_clusters.empty:
    regional_centres = wm.regional_centres_from_demand(demand, k=8)

demand[["name", "population", "was_relocated", "relocated_m"]]\
    .sort_values(["was_relocated", "relocated_m"], ascending=False)\
    .head(12)

In [ ]:
if not regional_clusters.empty:
    regional_clusters[["cluster_id", "population", "area_km2", "population_density"]]\
        .sort_values("population", ascending=False)
else:
    regional_centres[["centre_id", "demand_weight"]].sort_values("demand_weight", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
if not demand_areas.empty:
    demand_areas.plot(ax=ax, column="population_density", cmap="YlGnBu", alpha=0.65, edgecolor="#ffffff", linewidth=0.12, legend=True)
if not geology.empty:
    geology.plot(ax=ax, column="cost_factor", cmap="YlOrBr", alpha=0.14, edgecolor="none")
if water_crossings is not None and not water_crossings.empty:
    water_crossings.plot(ax=ax, color="#74b6d7", edgecolor="#2f6f95", alpha=0.42, linewidth=0.4)
if not flood_zones.empty:
    flood_zones.plot(ax=ax, color="#1b6f9c", edgecolor="#063b56", alpha=0.35)
if not centre_area.empty:
    centre_area.plot(ax=ax, facecolor="none", edgecolor="#111111", linewidth=2.2, linestyle="--", label="Stare Miasto")
if not regional_clusters.empty:
    regional_clusters.plot(ax=ax, facecolor="#f72585", alpha=0.10, edgecolor="#7a1f5c", linewidth=1.2, label="klastry regionalne")
demand_raw.plot(ax=ax, color="#555555", markersize=8, alpha=0.25, label="punkty reprezentatywne obszarów")
demand.plot(ax=ax, color="#d62828", markersize=10, alpha=0.45, label="popyt po relokacji")
regional_centres.plot(ax=ax, marker="x", color="#7a1f5c", markersize=70, linewidth=1.7, label="środki klastrów")
centres.plot(ax=ax, marker="*", color="#111111", markersize=120, label="centra wymuszone")

for _, row in demand[demand["was_relocated"]].iterrows():
    ax.plot(
        [row["original_geometry"].x, row.geometry.x],
        [row["original_geometry"].y, row.geometry.y],
        color="#d62828",
        linewidth=0.8,
        alpha=0.7,
    )

ax.set_title("Obszary popytu, Stare Miasto, rzeka i relokacja wag")
ax.set_axis_off()
ax.set_aspect("equal")
ax.legend(loc="lower left")
plt.show()

## Warstwy danych osobno

Poniższe mapy pokazują składniki danych osobno. Każda warstwa jest rysowana na podkładzie satelitarnym, żeby było widać realny kontekst miasta. Dla gęstości zaludnienia ciemniejszy obszar oznacza większe zagęszczenie, ale warstwa jest półprzezroczysta, więc zdjęcie satelitarne nadal zostaje czytelne.


In [ ]:
extent_for_maps = demand_areas if not demand_areas.empty else demand

layer_specs = [
    {
        "title": "Warstwa 1: gęstość zaludnienia",
        "layer": demand_areas,
        "column": "population_density",
        "cmap": "Greys",
        "alpha": 0.50,
        "edgecolor": "none",
    },
    {
        "title": "Warstwa 2: rzeki i wody powierzchniowe jako bariera komunikacyjna",
        "layer": water_crossings,
        "column": None,
        "color": "#48bfe3",
        "alpha": 0.50,
        "linewidth": 0.45,
    },
    {
        "title": "Warstwa 3: obszary zalewowe / zakazane, jeśli dostępne",
        "layer": flood_zones,
        "column": None,
        "color": "#ff3b30",
        "alpha": 0.35,
        "linewidth": 0.75,
    },
    {
        "title": "Warstwa 4: geologia jako mnożnik kosztu",
        "layer": geology,
        "column": "cost_factor",
        "cmap": "YlOrBr",
        "alpha": 0.42,
        "edgecolor": "#f5c542",
        "linewidth": 0.8,
    },
    {
        "title": "Warstwa 5: Stare Miasto jako wymuszone centrum",
        "layer": centre_area,
        "column": None,
        "color": "#ffffff",
        "alpha": 0.28,
        "edgecolor": "#111111",
        "linewidth": 2.0,
    },
    {
        "title": "Warstwa 6: regionalne klastry popytu",
        "layer": regional_clusters,
        "column": "population_density",
        "cmap": "magma_r",
        "alpha": 0.35,
        "edgecolor": "#ffffff",
        "linewidth": 1.2,
    },
]

for spec in layer_specs:
    fig, ax = wm.plot_data_layer(
        layer=spec["layer"],
        title=spec["title"],
        column=spec.get("column"),
        extent_layer=extent_for_maps,
        satellite=True,
        cmap=spec.get("cmap", "Greys"),
        alpha=spec.get("alpha", 0.55),
        color=spec.get("color", "#d62828"),
        edgecolor=spec.get("edgecolor", "#ffffff"),
        linewidth=spec.get("linewidth", 0.5),
        markersize=spec.get("markersize", 26),
    )
    plt.show()


## Optymalizacja wariantów jako problem NP-trudny

Właściwy model w projekcie to **orienteering problem**, czyli komiwojażer z nagrodami i budżetem. Każdy kandydat na stację ma nagrodę wynikającą z obsłużonego popytu, kolejność punktów działa jak w TSP, a limit długości/kosztu działa jak ograniczenie plecakowe. Problem jest NP-trudny, więc w notebooku używamy heurystyki: greedy insertion pod limitem długości, a potem lokalna poprawa 2-opt.

Funkcja celu: obsłużony popyt w promieniu dojścia do stacji, minus kara za przebieg przez obszary zakazane, minus kara za trudniejszą geologię, plus premia za sensowne przecięcie bariery rzecznej i za przesiadki z już zaplanowanymi liniami. Druga i trzecia linia są planowane na popycie resztkowym, czyli takim, którego poprzednie linie jeszcze dobrze nie obsłużyły.


In [ ]:
centre_point = centres.loc[centres["role"].eq("required_city_centre"), "geometry"].iloc[0]

candidate_sites = wm.candidate_station_sites(
    demand=demand,
    regional_centres=regional_centres,
    centres=centres,
    max_candidates=45,
)

candidate_sites[["candidate_id", "source", "name", "candidate_weight", "required"]].head(12)

Kandydaci na kotwice stacji są osobną warstwą wejściową algorytmu. Powstają z punktów popytu, centrów regionalnych i wymuszonego centrum. Na mapie poniżej można sprawdzić, czy algorytm faktycznie ma z czego wybierać w całym mieście.


In [ ]:
fig, ax = wm.plot_data_layer(
    layer=candidate_sites,
    title="Warstwa 7: kandydaci na kotwice stacji",
    column="candidate_weight",
    extent_layer=extent_for_maps,
    satellite=True,
    cmap="plasma_r",
    alpha=0.95,
    edgecolor="#ffffff",
    markersize=55,
)
plt.show()

### Formalizacja algorytmiczna

Niech `V` będzie zbiorem kandydatów na kotwice stacji, `c` wymaganym centrum, `I` punktami popytu, `w_i` wagą popytu, `d_uv` odległością między kandydatami, a `L = 23,1 km` limitem długości jednej linii.

Decyzje modelu:

- `z_v = 1`, jeżeli kandydat `v` został wybrany,
- `x_uv = 1`, jeżeli trasa przechodzi bezpośrednio z `u` do `v`,
- `y_i = 1` albo wartość ciągła w `[0, 1]`, jeżeli popyt `i` jest obsłużony przez wybrane stacje.

Cel: maksymalizować obsłużony popyt z karami za trudne fragmenty trasy:

`max sum(w_i * y_i) - kara_zalewowa - kara_geologiczna + premia_rzeka + premia_przesiadki`

Ograniczenia: trasa musi przejść przez Stare Miasto, musi mieć długość nie większą niż budżet, nie może tworzyć podtras oderwanych od głównej linii i musi zachować minimalne odstępy między kotwicami. To jest wariant orienteering / prize-collecting TSP, więc problem jest NP-trudny. W małej skali da się go policzyć dokładnie, ale dla realnej liczby kandydatów potrzebujemy heurystyk.


In [ ]:
wm.complexity_growth_table([5, 8, 10, 12, 15, 20, 30], max_selected=8)

### Exact vs heurystyka na małym podzbiorze

Dla kilku kandydatów możemy zrobić pełne przeszukanie: sprawdzamy możliwe podzbiory i kolejności przejścia przez centrum. To daje punkt odniesienia. Potem na tym samym podzbiorze uruchamiamy heurystykę greedy insertion + 2-opt i porównujemy wynik.


In [ ]:
exact_vs_heuristic, small_solutions = wm.compare_exact_and_heuristic(
    demand=demand,
    candidates=candidate_sites,
    centre=centre_point,
    forbidden=flood_zones,
    geology=geology,
    config=config,
    max_optional_candidates=8,
    max_selected_candidates=6,
)

exact_vs_heuristic.assign(
    served_share=lambda df: (100 * df["served_share"]).round(1),
    anchor_objective_score=lambda df: df["anchor_objective_score"].round(0),
    final_score=lambda df: df["final_score"].round(0),
    anchor_score_vs_exact=lambda df: (100 * df["anchor_score_vs_exact"]).round(1),
    forbidden_km=lambda df: df["forbidden_km"].round(2),
)

`anchor_objective_score` to wynik problemu NP-trudnego liczonego na wybranych kotwicach trasy. `final_score` to wynik po rozłożeniu pełnych 21 stacji na korytarzu. Te wartości mogą się różnić, bo kotwice sterują przebiegiem linii, a nie są jeszcze kompletną listą stacji.


In [ ]:
scenarios = wm.build_orienteering_scenarios(
    demand=demand,
    candidates=candidate_sites,
    centre=centre_point,
    forbidden=flood_zones,
    geology=geology,
    water_crossings=water_crossings,
    base_config=config,
)

summary = wm.scenario_summary_table(scenarios, demand)
summary.assign(
    length_km=lambda df: df["length_km"].round(1),
    served_share=lambda df: (100 * df["served_share"]).round(1),
    avg_geology_factor=lambda df: df["avg_geology_factor"].round(2),
    water_crossing_km=lambda df: df["water_crossing_km"].round(2),
    line_overlap_km=lambda df: df["line_overlap_km"].round(2),
    transfer_score=lambda df: df["transfer_score"].round(0),
    estimated_cost_mln=lambda df: df["estimated_cost_mln"].round(0),
)

W scenariuszach wieloliniowych algorytm nie planuje każdej nitki w izolacji. Po zbudowaniu pierwszej linii popyt w jej zasięgu jest zmniejszany jako popyt resztkowy, więc druga linia szuka obszarów nadal słabo obsłużonych. Dodatkowo kolejne linie dostają premię za sensowne przecięcie z wcześniejszymi liniami albo dojście w pobliże ich stacji, ale jednocześnie dostają karę za długie prowadzenie równolegle w tym samym korytarzu.


Poniżej widać dziennik decyzji heurystyki: który kandydat został wstawiony, na którą pozycję, jaki był przyrost wyniku i ile metrów trasy to dodało. To jest miejsce, w którym w raporcie można pokazać, że algorytm realnie rozwiązuje problem wyboru i kolejności stacji, a nie tylko rysuje linię.


In [ ]:
scenarios[selected_scenario if "selected_scenario" in globals() else "1"]["candidates"].head(15)

In [ ]:
selected_scenario = "3"  # zmień na "1", "2" albo "3"

fig, ax = wm.plot_scenario(
    demand=demand,
    forbidden=flood_zones,
    scenario=scenarios[selected_scenario],
    centres=centres,
    regional_centres=regional_centres,
    regional_clusters=regional_clusters,
    geology=geology,
    demand_areas=demand_areas,
    water_crossings=water_crossings,
    centre_area=centre_area,
    figsize=(10, 9),
)
plt.show()

In [ ]:
fig, ax = wm.plot_scenario_satellite(
    demand=demand,
    forbidden=flood_zones,
    scenario=scenarios[selected_scenario],
    centres=centres,
    regional_centres=regional_centres,
    regional_clusters=regional_clusters,
    geology=geology,
    demand_areas=demand_areas,
    water_crossings=water_crossings,
    centre_area=centre_area,
    figsize=(11, 10),
)
plt.show()

## Mapy satelitarne scenariuszy

Na finalnych mapach scenariuszy gęstość zaludnienia zostaje półprzezroczysta, a linie metra są rysowane jako mocne, nieprzezroczyste linie z czarną obwódką. Dzięki temu linia nie ginie na zdjęciu satelitarnym ani na ciemniejszych obszarach zagęszczenia.


In [ ]:
for scenario_id, scenario in scenarios.items():
    fig, ax = wm.plot_scenario_satellite(
        demand=demand,
        forbidden=flood_zones,
        scenario=scenario,
        centres=centres,
        regional_centres=regional_centres,
        regional_clusters=regional_clusters,
        geology=geology,
        demand_areas=demand_areas,
        water_crossings=water_crossings,
        centre_area=centre_area,
        figsize=(9, 8),
    )
    ax.set_title(f"Scenariusz {scenario_id}: {scenario['name']}")
    plt.show()

## Kotwice wybrane przez algorytm

Kotwice to punkty, które heurystyka wybrała i ułożyła w trasę. Finalne stacje są potem rozłożone równomiernie na korytarzu o długości 23,1 km, żeby zachować porównywalność z warszawską M1.


In [ ]:
anchors = pd.concat([line["anchors"] for line in scenarios[selected_scenario]["lines"]], ignore_index=True)
anchors_wgs = gpd.GeoDataFrame(anchors, geometry="geometry", crs=anchors.crs).to_crs(4326)
anchors_wgs["lon"] = anchors_wgs.geometry.x
anchors_wgs["lat"] = anchors_wgs.geometry.y
anchors_wgs[["line_id", "anchor_order", "candidate_id", "name", "source", "lon", "lat"]].head(40)

## Tabela stacji dla wybranego scenariusza

Stacje są rozmieszczone równomiernie na linii, bo trzymamy benchmark M1. W wersji kolejnej warto dodać korektę położenia stacji do najbliższych lokalnych maksimów popytu, ale z ograniczeniem minimalnego i maksymalnego odstępu między stacjami.


In [ ]:
lines, stations = wm.scenario_geodataframes(scenarios[selected_scenario])
stations_wgs = stations.to_crs(4326).copy()
stations_wgs["lon"] = stations_wgs.geometry.x
stations_wgs["lat"] = stations_wgs.geometry.y

stations_wgs[["line_id", "station_code", "distance_m", "lon", "lat"]].head(30)

## Napotkane problemy i rozwiązania

1. Problem punktów zamiast obszarów: pierwsza wersja traktowała popyt jako punkty. Rozwiązanie: popyt jest teraz wczytywany i rysowany jako obszary demograficzne SIP, a punkty reprezentatywne są używane tylko w algorytmie odległościowym.
2. Problem rzeki jako zakazu: woda powierzchniowa nie powinna blokować metra tak jak brak gruntu dla zwykłej drogi. Rozwiązanie: rzeka jest osobną warstwą bariery komunikacyjnej; przecięcie tej bariery może dostać premię, bo tworzy nowe połączenie między brzegami.
3. Problem zalewowości: obszary ryzyka nie powinny przyciągać stacji, ale popyt z ich okolic nie powinien znikać. Rozwiązanie: wagi popytu z obszarów zakazanych są przesuwane do najbliższego wolnego miejsca.
4. Problem izolowanych i nakładających się linii: kolejne nitki metra nie mogą być planowane tak, jakby wcześniejsze nie istniały, ale nie powinny też leżeć na sobie. Rozwiązanie: po każdej linii zmniejszamy popyt resztkowy, premiujemy sensowne przesiadki i karzemy długi przebieg w buforze już zaplanowanego korytarza.
5. Problem NP-trudności: pełne przeszukanie kombinacji i kolejności stacji szybko staje się nierealne. Rozwiązanie: dla małej próbki pokazujemy brute force, a dla pełnego problemu używamy heurystyki greedy insertion + 2-opt.
6. Problem interpretacji wyniku: kotwice algorytmu nie są jeszcze finalną listą stacji. Rozwiązanie: rozdzielamy `anchor_objective_score` od `final_score`, czyli wynik wyboru korytarza od wyniku po rozłożeniu 21 stacji.


## Jak rozwijać model

Najbliższe sensowne kroki:

1. Pobrać demografię SIP Wrocławia 1998-2025 i użyć rejonów statystycznych jako podstawowego popytu.
2. Pobrać obszary zagrożenia powodziowego MZP z Hydroportalu i zapisać jako `data/raw/flood_zones.geojson` albo shapefile.
3. Uzupełnić `geology` o realny mnożnik kosztu: odwierty, grunty, poziom wód gruntowych, tunele pod rzeką.
4. Dodać aktualne węzły transportowe: dworce, pętle tramwajowe, duże przesiadki autobusowe, uczelnie, szpitale, duże osiedla.
5. Zastąpić linie proste grafem kandydatów: korytarze uliczne, kolejowe, podziemne i zakazane poligony jako koszt przejścia.
6. Dodać walidację: ile mieszkańców ma stację w 600, 800 i 1000 m oraz ile obecnych podróży tramwaj/autobus może zostać przejętych.
